# 02 — Baseline model inference

**Objective:** measure unmodified `HuggingFaceTB/SmolLM2-360M-Instruct` on held-out questions **before fine-tuning**. The CLI is the source of truth; this notebook only runs it and visualizes its recorded output.

**Status:** not executed against the real model in the initial isolated environment (Transformers/weights/network unavailable). Do not present illustrative or mock measurements as real. Use a Python 3.11 environment with `pip install -e '.[train]'` and `pip install psutil pandas matplotlib`. Allow ~1.5 GB just for fp32 model parameters, plus runtime overhead.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

root = Path.cwd()
if not (root / "pyproject.toml").exists():
    root = root.parent
report_path = root / "outputs/baseline.json"
print("Repository:", root)
print("Baseline output:", report_path)

Repository: /mnt/data/tinydevassist-work-m5/tiny-llm-hf
Baseline output: /mnt/data/tinydevassist-work-m5/tiny-llm-hf/outputs/baseline.json


## Run the baseline (explicit opt-in)

Set `RUN_BASELINE = True` to download the unmodified model if needed, generate responses to the held-out test prompts, and save the report. The default `False` avoids an unexpected network download. For reproducibility, replace `main` with a pinned model commit SHA.

In [2]:
RUN_BASELINE = False
if RUN_BASELINE:
    subprocess.run([sys.executable, str(root / "prepare_data.py"), "--seed", "42"],
                   cwd=root, check=True)
    subprocess.run([sys.executable, str(root / "inference.py"),
                    "--revision", "main", "--max-samples", "6", "--device", "cpu"],
                   cwd=root, check=True)
else:
    print("Skipped: set RUN_BASELINE=True when model dependencies and weights are available.")

Skipped: set RUN_BASELINE=True when model dependencies and weights are available.


In [3]:
if report_path.exists():
    report = json.loads(report_path.read_text(encoding="utf-8"))
    if report.get("status") != "measured":
        raise ValueError("Not a measured baseline report")
    print("Model:", report["model_id"])
    print("Resolved revision:", report["resolved_revision"])
    print("Device:", report["device"])
    print("Summary:", report["summary"])
    import pandas as pd
    display(pd.DataFrame([{
        "id": r["id"], "prompt": r["prompt"], "response": r["response"],
        "latency_s": r["metrics"]["latency_seconds"],
        "generated_tokens": r["metrics"]["generated_tokens"],
    } for r in report["results"]]))
else:
    print("No real baseline report exists. Run the CLI or opt-in cell first.")

No real baseline report exists. Run the CLI or opt-in cell first.


## Review questions

1. Are the generated answers technically correct, actionable, and appropriately cautious?
2. Which prompts produce hallucinations or omissions? Record **actual** good and bad generations after running.
3. What is the median latency on your machine? Compare it only with measurements on equivalent hardware/settings.
4. RSS before/after are snapshots, **not peak CPU memory**; CUDA peak allocated is measured only when CUDA is used.

The current 6-question held-out set is too small to claim broad improvement. Later evaluation will add independent test cases and a transparent scoring rubric.